# mlmodels end-goal API, pillar 2 -- continuous progress feedback

**Goal (L3f.2).** Surface training progress **live, per epoch, as it runs** -- not only
after `fit` returns -- and in **interpretable units** (validation RMSE in solar masses),
through a first-class injected `LiveProgress` callback.

Same notebook-first, hand-gluing workflow. Builds on `ns_mlmodels-split-train-val.ipynb`
(the split + validation pillar); here the emphasis is the **feedback loop**.

> **Headless note.** When this notebook is executed non-interactively (nbconvert), the
> per-epoch lines below form the training log. Run interactively in JupyterLab and the
> same `print(..., flush=True)` lines appear one-by-one as each epoch finishes -- the
> live feedback. The callback is the reusable piece; the display surface is the notebook.

## 0. Config / env seam (same as the companion notebooks)

In [ ]:
import os
from pathlib import Path


def project_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise RuntimeError("pyproject.toml not found above cwd")


ROOT = project_root()
VAR = ROOT / "var" / "data" / "surrogate_models"
CKPT_DIR = VAR / "checkpoints" / "nb-live"
os.environ["SURROGATE_MODELS__DATASETS__PATH"] = str(VAR / "datasets")
os.environ["SURROGATE_MODELS__DATASETS__NEUTRON_STARS_SOURCE"] = str(
    ROOT / "data" / "neutron-stars" / "neutron-stars.dat"
)
os.environ["SURROGATE_MODELS__MLMODELS__CHECKPOINT_DIR"] = str(CKPT_DIR)
os.environ["SURROGATE_MODELS__LOGGING__FILE"] = str(VAR / "log" / "surrogate_models.log")
print("checkpoint dir:", CKPT_DIR)

In [ ]:
%matplotlib inline
from functools import partial
from typing import Any

import lightning
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from surrogate_models import load_neutron_stars
from surrogate_models.mlmodels.application import TrainRun, handle_train_run
from surrogate_models.mlmodels.domain import (
    Checkpoint,
    ConfiguredRun,
    make_holdout_spec,
    make_model_identity,
)
from surrogate_models.mlmodels.infrastructure import (
    RunManifest,
    structural_fingerprint,
    write_run_manifest,
)
from surrogate_models.railway_adts import fmap_error, safe

torch.manual_seed(0)
_ = lightning.seed_everything(0, verbose=False)

## 1. Data, split, and model (condensed -- proven in the companion notebook)

The repeated setup here (tensor-ization, seeded split) is itself a harvest signal --
already flagged as a candidate to move into the `datasets` BC. `Y_STD` is kept because
the callback converts standardized MSE back to solar masses for readable feedback.

In [ ]:
df = load_neutron_stars()
FEATURES, TARGET = ["rhoc", "Pc"], "M"
sample = df[[*FEATURES, TARGET]].dropna().sample(n=3000, random_state=0)
x_raw = torch.tensor(sample[FEATURES].to_numpy(), dtype=torch.float32)
y_raw = torch.tensor(sample[[TARGET]].to_numpy(), dtype=torch.float32)
X = (x_raw - x_raw.mean(0)) / x_raw.std(0)
y_mean, y_std = y_raw.mean(0), y_raw.std(0)
Y = (y_raw - y_mean) / y_std
Y_STD = float(y_std)

SPLIT = make_holdout_spec(0.7, 0.15, 0.15, 0).unwrap()
perm = torch.randperm(X.shape[0], generator=torch.Generator().manual_seed(0))
n_train = int(SPLIT.train_fraction * X.shape[0])
n_val = int(SPLIT.val_fraction * X.shape[0])
idx_train, idx_val = perm[:n_train], perm[n_train:n_train + n_val]
train_loader = DataLoader(TensorDataset(X[idx_train], Y[idx_train]), batch_size=64, shuffle=True)
val_loader = DataLoader(TensorDataset(X[idx_val], Y[idx_val]), batch_size=64, shuffle=False)


class SplitSurrogate(lightning.LightningModule):
    '''A 2->8->1 MLP with per-epoch validation -- notebook model only.'''

    MODEL_NAME = "ns-mlp-live"
    MODEL_VERSION = "0.1.0"

    def __init__(self, n_features: int, learning_rate: float, optimizer_name: str) -> None:
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_features, 8), nn.ReLU(), nn.Linear(8, 1))
        self.learning_rate = learning_rate
        self.optimizer_name = optimizer_name

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        out: torch.Tensor = self.net(features)
        return out

    def training_step(self, *args: Any, **kwargs: Any) -> torch.Tensor:
        features, targets = args[0]
        loss: torch.Tensor = nn.functional.mse_loss(self.net(features), targets)
        self.log("train_loss", loss, on_step=False, on_epoch=True, logger=False)
        return loss

    def validation_step(self, *args: Any, **kwargs: Any) -> torch.Tensor:
        features, targets = args[0]
        loss: torch.Tensor = nn.functional.mse_loss(self.net(features), targets)
        self.log("val_loss", loss, on_step=False, on_epoch=True, logger=False)
        return loss

    def configure_optimizers(self) -> torch.optim.Optimizer:
        match self.optimizer_name:
            case "adam":
                return torch.optim.Adam(self.parameters(), lr=self.learning_rate)
            case _:
                return torch.optim.SGD(self.parameters(), lr=self.learning_rate)


def build_model(config: Any) -> SplitSurrogate:
    return SplitSurrogate(len(FEATURES), config.learning_rate, config.optimizer)

## 2. The `LiveProgress` callback -- feedback as each epoch finishes

On every validation-epoch end it records `(epoch, train_MSE, val_MSE, val_RMSE_Msun)`
and **prints one line immediately** (`flush=True`), so progress streams as it happens.
`val_RMSE_Msun = sqrt(val_MSE) * Y_STD` converts the standardized loss back to solar
masses (the target was scaled by `Y_STD`). The Lightning sanity-check pass is skipped
because `train_loss` is not yet in the metrics then.

In [ ]:
class LiveProgress(lightning.Callback):
    '''Stream per-epoch train/val loss + val RMSE (Msun) as training runs.'''

    def __init__(self, y_std: float) -> None:
        super().__init__()
        self.y_std = y_std
        self.rows: list[tuple[int, float, float, float]] = []

    def on_validation_epoch_end(
        self, trainer: lightning.Trainer, pl_module: lightning.LightningModule
    ) -> None:
        m = trainer.callback_metrics
        if "train_loss" not in m or "val_loss" not in m:
            return  # skip Lightning's pre-training sanity-check pass
        epoch = trainer.current_epoch
        train_mse, val_mse = float(m["train_loss"]), float(m["val_loss"])
        val_rmse = (val_mse ** 0.5) * self.y_std
        self.rows.append((epoch, train_mse, val_mse, val_rmse))
        print(
            f"epoch {epoch:2d} | train MSE {train_mse:.4f} | "
            f"val MSE {val_mse:.4f} | val RMSE {val_rmse:.4f} Msun",
            flush=True,
        )

## 3. Train with live feedback

The `LiveProgress` callback is injected into the same adapter closure as the loaders and
handed to the `Trainer`. Watch the per-epoch lines stream out below.

In [ ]:
@safe(Exception, fmap_error(lambda cause: cause, code="TRAINING_FAILED"))
def save_with_feedback(
    checkpoint_dir: Path,
    train_dl: DataLoader,
    val_dl: DataLoader,
    monitor: LiveProgress,
    run: ConfiguredRun,
) -> Checkpoint:
    '''Fit with train+val loaders + a live-feedback callback, then persist ckpt+manifest.'''
    model = build_model(run.config)
    trainer = lightning.Trainer(
        max_epochs=run.config.max_epochs, accelerator="cpu", devices=1, logger=False,
        enable_checkpointing=False, enable_progress_bar=False, enable_model_summary=False,
        callbacks=[monitor],
    )
    trainer.fit(model, train_dl, val_dl)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    target = checkpoint_dir / f"{run.run_id}.ckpt"
    torch.save(model.state_dict(), target)
    write_run_manifest(checkpoint_dir, RunManifest(
        str(run.run_id), model.MODEL_NAME, model.MODEL_VERSION,
        run.config.max_epochs, run.config.learning_rate, run.config.batch_size,
        run.config.optimizer, structural_fingerprint(model.state_dict())))
    return Checkpoint(str(target))


monitor = LiveProgress(Y_STD)
cmd = TrainRun("ns-live", max_epochs=20, learning_rate=0.05, batch_size=64, optimizer="adam")
save = partial(save_with_feedback, CKPT_DIR, train_loader, val_loader, monitor)
trained = handle_train_run(save_trained_run=save, cmd=cmd).unwrap()
print("done:", trained.run_id)

## 4. The recorded history as a curve

In [ ]:
epochs = [r[0] for r in monitor.rows]
plt.figure(figsize=(6, 3.5))
plt.plot(epochs, [r[3] for r in monitor.rows], marker="o", label="val RMSE (Msun)")
plt.xlabel("epoch")
plt.ylabel("val RMSE (Msun)")
plt.title("validation error per epoch")
plt.legend()
plt.tight_layout()
plt.show()

## Findings + harvest note

- **Continuous feedback works with the injected-callback seam** -- `LiveProgress` streams
  interpretable per-epoch metrics with no `src` change, same as the loaders. The callback
  is the reusable unit.
- **Harvest candidate (sharpened):** a first-class progress callback belongs in
  `mlmodels` once its shape settles -- but note it needs the de-standardization constant
  (`Y_STD`) to report physical units, which reinforces the earlier flag that **the
  standardizer must be persisted with the model** (today it lives only in the notebook).
  Recording metrics + the standardizer together is the natural next harvest.
- **Next pillar:** L3f.3 checkpoints + resume (per-epoch checkpoints, load-and-continue),
  where the fork-2 `trainer.save_checkpoint` + `materialize_model` format fix lands.